# Inventory coverage and replenishment review

This notebook uses observed sales and the latest available stock value per product. The example reorder calculation explicitly assumes a 14-day lead time and one additional week of demand as safety stock; adjust these assumptions to match operations.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

root = Path.cwd()
if not (root / "datasets").exists(): root = root.parent
df = pd.read_csv(root / "datasets" / "autods_demo_sales.csv", parse_dates=["Order Date"])
required = {"Order Date", "Product Name", "Quantity", "Stock Quantity"}
missing = required - set(df.columns)
if missing: raise ValueError(f"Inventory analysis requires these columns: {sorted(missing)}")
df = df.dropna(subset=["Order Date", "Product Name"])
end_date = df["Order Date"].max().normalize()
print(f"Dataset end date: {end_date.date()} | Product count: {df['Product Name'].nunique()}")

In [ ]:
snapshot = (df.sort_values("Order Date").groupby("Product Name", as_index=False)
            .tail(1).set_index("Product Name")["Stock Quantity"])
recent = df[df["Order Date"] > end_date - pd.Timedelta(days=30)]
demand_30d = recent.groupby("Product Name")["Quantity"].sum()
inventory = pd.DataFrame({"stock_snapshot": snapshot, "units_sold_last_30d": demand_30d}).fillna({"units_sold_last_30d": 0})
inventory["daily_demand"] = inventory["units_sold_last_30d"] / 30
inventory["coverage_days"] = np.where(inventory["daily_demand"] > 0, inventory["stock_snapshot"] / inventory["daily_demand"], np.inf)
lead_time_days = 14
safety_stock_days = 7
inventory["reorder_point_units"] = np.ceil(inventory["daily_demand"] * (lead_time_days + safety_stock_days))
inventory["suggested_reorder_units"] = np.maximum(0, inventory["reorder_point_units"] - inventory["stock_snapshot"])
inventory["signal"] = np.select(
    [inventory["daily_demand"].eq(0), inventory["coverage_days"] < lead_time_days, inventory["coverage_days"] < 30, inventory["coverage_days"] > 180],
    ["No recent demand", "Review stockout risk", "Monitor", "Review overstock"], default="Coverage appears adequate")
display(inventory.sort_values("coverage_days").replace([np.inf], np.nan).round(2))
print(f"Assumptions: {lead_time_days}-day lead time; {safety_stock_days} additional safety-stock days. No holding or order costs are available for EOQ.")